# ARC NeuroGolf static ONNX solver

Reference layout adapted from the uploaded fill/additive-marking notebook. The task-specific modelling cell uses a semantic feature-tree or a symbolic reflection builder, not raw output-template lookup.

In [1]:
!rm -rf /kaggle/working/*
%reset -f

In [2]:
COMPETITION = '/kaggle/input/competitions/neurogolf-2026'

In [3]:
import importlib.util, subprocess, sys
missing=[p for p in ['onnx','onnxruntime','onnxscript','torch','numpy'] if importlib.util.find_spec(p) is None]
if missing:
    subprocess.check_call([sys.executable,'-m','pip','install','-q',*missing])
print('dependencies ok')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 44.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 722.0/722.0 kB 33.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 166.8/166.8 kB 10.8 MB/s eta 0:00:00
dependencies ok


In [4]:
import json, os, time, hashlib, zipfile,  csv, base64
import glob, sys,math, random, collections,io,shutil
from pathlib import Path
import numpy as np
import torch
import onnx
import onnxruntime as ort
import torch, torch.nn as nn, torch.nn.functional as F
from collections import defaultdict
from onnx import shape_inference

In [5]:
TASK_ID='task197'; CH=10; H=W=30
FORBIDDEN={'Loop','Scan','NonZero','Unique','Script','Function'}
TASK_PATH=Path('/mnt/data/task197.json')
if not TASK_PATH.exists() and Path('/kaggle/input').exists():
    m=list(Path('/kaggle/input').rglob('task197.json'))
    TASK_PATH=m[0] if m else TASK_PATH
OUT_DIR=Path('/kaggle/working') if Path('/kaggle/working').exists() else Path('/mnt/data/task197_canvasmask_30x30')
OUT_DIR.mkdir(parents=True,exist_ok=True)
ONNX_PATH=OUT_DIR/'task197.onnx'
SUBMISSION_PATH=OUT_DIR/'submission.zip'
print('TASK_PATH=',TASK_PATH)
print('OUT_DIR=',OUT_DIR)

TASK_PATH= /kaggle/input/competitions/neurogolf-2026/task197.json
OUT_DIR= /kaggle/working


In [6]:
class Task197CanvasMaskModel(nn.Module):
    def forward(self,x):
        active=(torch.sum(x,dim=1,keepdim=True)>0.5).to(x.dtype)
        fg=torch.sum(x[:,1:,:,:],dim=1,keepdim=True)
        row_fg=fg.sum(dim=3,keepdim=True)
        max_row=row_fg.max(dim=2,keepdim=True).values
        header_row=((torch.abs(row_fg-max_row)<0.25) * (max_row>0.5)).to(x.dtype)
        header=(x*header_row).sum(dim=2,keepdim=True)
        fg_out=[]
        for out_color in range(1,CH):
            acc=x.new_zeros((x.shape[0],1,H,W))
            for header_color in range(1,CH):
                hmask=header[:,header_color:header_color+1,:,:]
                row_map=(x[:,out_color:out_color+1,:,:]*hmask).sum(dim=3,keepdim=True)
                row_map=(row_map>0.25).to(x.dtype)
                acc=acc+row_map*hmask
            fg_out.append((acc>0.25).to(x.dtype)*active)
        fg_cat=torch.cat(fg_out,dim=1)
        bg=(fg_cat.sum(dim=1,keepdim=True)<0.5).to(x.dtype)*active
        return torch.cat([bg,fg_cat],dim=1)
model=Task197CanvasMaskModel().eval()
print(model)

Task197CanvasMaskModel()


In [7]:
task=json.loads(TASK_PATH.read_text()) if TASK_PATH.exists() else None
print({k:len(task.get(k,[])) for k in ['train','test','arc-gen']} if task else 'no local json')

def grid_to_tensor(grid):
    arr=np.asarray(grid,dtype=np.int64)
    x=np.zeros((1,CH,H,W),dtype=np.float32)
    h,w=arr.shape
    for c in range(CH):
        x[0,c,:h,:w]=(arr==c).astype(np.float32)
    return x

def padded_expected(grid):
    arr=np.asarray(grid,dtype=np.int64)
    y=np.zeros((H,W),dtype=np.int64)
    y[:arr.shape[0],:arr.shape[1]]=arr
    return y

def pred_grid(session,grid):
    y=session.run(None,{session.get_inputs()[0].name:grid_to_tensor(grid)})[0]
    return y.argmax(axis=1)[0].astype(np.int64)

def eval_examples(session,examples):
    ok=0; wrong=[]
    for i,ex in enumerate(examples):
        good=np.array_equal(pred_grid(session,ex['input']), padded_expected(ex['output']))
        ok+=int(good)
        if not good and len(wrong)<5: wrong.append(i)
    return {'exact':ok,'total':len(examples),'wrong_sample':wrong}

{'train': 4, 'test': 1, 'arc-gen': 262}


In [8]:
dummy=np.zeros((1,CH,H,W),dtype=np.float32)
dummy[0,0,:14,:10]=1.0
torch.onnx.export(model,torch.from_numpy(dummy),str(ONNX_PATH),input_names=['input'],output_names=['output'],opset_version=17,do_constant_folding=True,dynamic_axes=None,dynamo=False)
m=onnx.load(str(ONNX_PATH)); onnx.checker.check_model(m); m=shape_inference.infer_shapes(m); onnx.save(m,str(ONNX_PATH))
print('saved',ONNX_PATH,'bytes',ONNX_PATH.stat().st_size)

/tmp/ipykernel_16/2490701245.py:3: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  torch.onnx.export(model,torch.from_numpy(dummy),str(ONNX_PATH),input_names=['input'],output_names=['output'],opset_version=17,do_constant_folding=True,dynamic_axes=None,dynamo=False)


saved /kaggle/working/task197.onnx bytes 123895


In [9]:
m=onnx.load(str(ONNX_PATH)); onnx.checker.check_model(m)
ops=collections.Counter(n.op_type for n in m.graph.node)
forbidden=sorted(set(ops)&FORBIDDEN)
empty=[(n.name,n.op_type,list(n.input)) for n in m.graph.node if any(i=='' for i in n.input)]
def dims(vi): return [d.dim_value if d.HasField('dim_value') else None for d in vi.type.tensor_type.shape.dim]
bad=[]
for vi in list(m.graph.input)+list(m.graph.output)+list(m.graph.value_info):
    if vi.type.HasField('tensor_type'):
        ds=dims(vi)
        if len(ds)>0 and any(d in (None,0) for d in ds): bad.append((vi.name,ds))
assert not forbidden, forbidden
assert not empty, empty
assert not bad[:1], bad[:3]
assert ONNX_PATH.stat().st_size < 1_440_000
print('ops',dict(sorted(ops.items())))
print('static/validator checks ok')

ops {'Abs': 1, 'Add': 81, 'And': 1, 'Cast': 93, 'Concat': 2, 'Constant': 182, 'Greater': 92, 'Less': 2, 'Mul': 173, 'ReduceMax': 1, 'ReduceSum': 86, 'Slice': 19, 'Sub': 1}
static/validator checks ok


In [10]:
session=ort.InferenceSession(str(ONNX_PATH),providers=['CPUExecutionProvider'])
if task:
    summary={}
    for split in ['train','test','arc-gen']:
        summary[split+'_raw_padded']=eval_examples(session,task.get(split,[]))
    print(json.dumps(summary,indent=2))
    assert summary['train_raw_padded']['exact']==summary['train_raw_padded']['total']
    assert summary['test_raw_padded']['exact']==summary['test_raw_padded']['total']
    if 'arc-gen' in task:
        assert summary['arc-gen_raw_padded']['exact']==summary['arc-gen_raw_padded']['total']

{
  "train_raw_padded": {
    "exact": 4,
    "total": 4,
    "wrong_sample": []
  },
  "test_raw_padded": {
    "exact": 1,
    "total": 1,
    "wrong_sample": []
  },
  "arc-gen_raw_padded": {
    "exact": 262,
    "total": 262,
    "wrong_sample": []
  }
}


In [11]:
with zipfile.ZipFile(SUBMISSION_PATH,'w',zipfile.ZIP_DEFLATED) as z:
    z.write(ONNX_PATH,arcname='task197.onnx')
print('wrote',SUBMISSION_PATH)
print('zip contents:',zipfile.ZipFile(SUBMISSION_PATH).namelist())

wrote /kaggle/working/submission.zip
zip contents: ['task197.onnx']
